In [3]:
%%capture --no-stderr
%pip install -U langgraph
%pip install -U langchain-perplexity



In [5]:
import importlib.metadata
print(importlib.metadata.version("langgraph"))


1.0.3


In [5]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage

messages = [AIMessage(content=f"You want to research about the stalk market?", name="Model")]
messages.append(HumanMessage(content=f"Yes, that's right.",name="Akshita"))
messages.append(AIMessage(content=f"Great, what would you like to learn about.", name="Model"))
messages.append(HumanMessage(content=f"I want to learn about the best time to invest in TATA.", name="Akshita"))

for m in messages:
    m.pretty_print()

================================== Ai Message ==================================
Name: Model

You want to research about the stalk market?
================================ Human Message =================================
Name: Akshita

Yes, that's right.
================================== Ai Message ==================================
Name: Model

Great, what would you like to learn about.
================================ Human Message =================================
Name: Akshita

I want to learn about the best time to invest in TATA.


In [5]:
import os, getpass

def set_key(name: str):
    if not os.getenv(name):
        os.environ[name] = getpass.getpass(f"{name}: ")

set_key("PPLX_API_KEY")


PPLX_API_KEY:  ········


In [1]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("PPLX_API_KEY")


PPLX_API_KEY:  ········


In [4]:
from langchain_perplexity import ChatPerplexity
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatPerplexity(
    model="sonar-pro"
)

messages = [
    SystemMessage(content="You are a helpful financial research assistant."),
    HumanMessage(content="You want to research about the stock market?", name="Akshita"),
    AIMessage(content="Yes, that's right.", name="Model"),
    HumanMessage(content="I want to learn about the best time to invest in TATA.", name="Akshita"),
]

response = llm.invoke(messages)
print(response.content)


Based on the latest available information up to November 2025, here’s what you need to know about the best time to invest in Tata Group stocks:

### Current Market Situation (2025)
- **Sharp Decline in Market Cap:** The Tata Group has seen a significant drop in its combined market capitalization, losing around $73–$120 billion since late 2024. This is due to a mix of sectoral headwinds, leadership uncertainty, and global challenges.
- **Key Drivers of Decline:**
  - **TCS (Tata Consultancy Services):** Struggles with slowing growth, visa/tariff issues in Western markets, and margin pressures.
  - **Tata Motors:** Weakness in Jaguar Land Rover (JLR) sales, cyberattack, and broader auto sector slowdown.
  - **Trent Ltd, Tejas Networks, Tata Technologies:** Sharp corrections (40–50% from peaks) due to sectoral and execution issues.
  - **Governance Concerns:** Internal leadership disputes and board-level uncertainty have further dampened investor sentiment.

### Signs of Potential Recover

In [6]:
{
    "token_usage": {
        "prompt_tokens": 67,
        "completion_tokens": 323,
        "total_tokens": 390
    },
    "model_name": "sonar-pro",
    "finish_reason": "stop",
    "id": "response-001"
}


{'token_usage': {'prompt_tokens': 67,
  'completion_tokens': 323,
  'total_tokens': 390},
 'model_name': 'sonar-pro',
 'finish_reason': 'stop',
 'id': 'response-001'}

Let's showcase a simple example of tool calling!
 
The `multiply` function is our tool.

In [8]:
def mul(a: int, b: int) -> int:
    return a * b

llm_tools = llm.bind_tools([mul])


NotImplementedError: 

In [9]:
tool_call = llm_with_tools.invoke([HumanMessage(content=f"What is 2 multiplied by 3", name="Akshita")])

NameError: name 'llm_with_tools' is not defined

In [9]:
tool_call.tool_calls

[{'name': 'multiply',
  'args': {'a': 2, 'b': 3},
  'id': 'call_6Zk3hXVbK7pHTPlGdopY9o5b',
  'type': 'tool_call'}]

In [10]:
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage

class MessagesState(TypedDict):
    messages: list[AnyMessage]

In [11]:
from typing import Annotated
from langgraph.graph.message import add_messages

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [12]:
from langgraph.graph import MessagesState

class MessagesState(MessagesState):
    pass

To go a bit deeper, we can see how the `add_messages` reducer works in isolation.

In [10]:
initial_messages = [AIMessage(content="Hello! How can I assist you?", name="Model"),
                    HumanMessage(content="I'm looking for information on stock market.", name="Lance")
                   ]

new_message = AIMessage(content="Sure, I can help with that. What specifically are you interested in?", name="Model")

add_messages(initial_messages , new_message)

NameError: name 'add_messages' is not defined

In [11]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
    def tool_calling_llm(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}
builder = StateGraph(MessagesState)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_edge(START, "tool_calling_llm")
builder.add_edge("tool_calling_llm", END)
graph = builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

IndentationError: unexpected indent (1185701409.py, line 3)

In [15]:
messages = graph.invoke({"messages": HumanMessage(content="Hello!")})
for m in messages['messages']:
    m.pretty_print()

================================ Human Message =================================

Hello!
================================== Ai Message ==================================

Hello! How can I assist you today?


The LLM chooses to use a tool when it determines that the input or task requires the functionality provided by that tool.

In [16]:
def pretty_print_with_flavour(msg):
    try:
        msg.pretty_print()
    except Exception:
        print('->', getattr(msg, 'name', 'Unknown'), ':', getattr(msg, 'content', str(msg)))
    print('✨ That was a thoughtful reply!')

messages = graph.invoke({"messages": HumanMessage(content="Multiply 2 and 3")})
for m in messages['messages']:
    pretty_print_with_flavour(m)

================================ Human Message =================================

Multiply 2 and 3
✨ That was a thoughtful reply!
================================== Ai Message ==================================
Tool Calls:
  multiply (call_OpxqZUzkH0QWuMZK1ZDU0Dk8)
 Call ID: call_OpxqZUzkH0QWuMZK1ZDU0Dk8
  Args:
    a: 2
    b: 3
✨ That was a thoughtful reply!
